# Workflow Patterns: Structured Output, Streaming, and Interrupts

The previous notebooks covered agent construction, tool use, and memory. This notebook focuses on three workflow patterns that matter once you move beyond simple request-and-response examples.

> [!NOTE]
> This is where the optional extensions begin. Notebooks 1 to 4 are the core path (the agent loop by hand, then via LangChain, then via LangGraph). Notebooks 5 to 7 build on that core and can be taken later, in any order. Cover them if you have time after the core notebooks.

The examples use three small workflows:

- an incident triage agent that classifies an operational alert into a typed data structure
- a deployment brief graph that streams progress updates as it runs
- a change approval graph that pauses and waits for a human decision before continuing

## Learning Objectives

At the end of this notebook, you should be able to:

- Implement structured output so an agent returns typed, validated data instead of prose.
- Apply streaming to surface each step of a graph while it runs.
- Build an interrupt so a graph pauses for a human decision and resumes from the same point.
- Compare when each of these patterns fits a real workflow.

A basic agent usually returns one text response at the end. That is fine for demos, but real workflows often need something more specific.

- a downstream system may need **structured data** instead of a paragraph
- a user may need **progress updates** while work is still running
- a risky action may need a **human decision** before the workflow can continue

The sections below isolate one pattern for each case.

## Workflow sketch

```mermaid
flowchart LR
    U["User request"] --> S["Structured output"]
    S --> T["Stream progress"]
    T --> I{"Review needed?"}
    I -->|yes| H["interrupt()"]
    H --> R["Command(resume=...)"]
    R --> F["Finalize state"]
    I -->|no| F
```

Each pattern is isolated in its own section so the behaviors can be compared directly.

---

## Quick reference: `TypedDict` and `Pydantic`

This notebook uses two data-shape tools:

- `TypedDict` describes the graph's internal state as it moves between nodes. It is useful for clarity and type checking, but it does not validate values at runtime.
- `Pydantic` describes structured data that must be validated before application code uses it. Here, it is used for model output that should come back with fixed fields and types.

In short: `TypedDict` is for workflow state; `Pydantic` is for validated structured results.

---

## Import the building blocks

The following imports provide the pieces used by all three examples.

- LangChain helpers for the model and structured output
- LangGraph pieces for graphs, streaming, and interrupts
- `TypedDict` for graph state and `Pydantic` for validated structured output

In [ ]:
from typing import Any, Literal, TypedDict, cast  # type hints

from dotenv import load_dotenv  # read secret keys from a .env file
from IPython.display import Image, display  # show images inside the notebook
from langchain.agents import create_agent  # build a LangChain agent
from langchain.chat_models import (
    init_chat_model,
)  # choose and connect to an AI model (e.g. GPT)
from langgraph.checkpoint.memory import (
    InMemorySaver,
)  # save agent state to enable memory
from langgraph.config import (
    get_stream_writer,
)  # emit custom events during graph execution
from langgraph.graph import (
    END,
    START,
    StateGraph,
)  # define the steps and flow of the agent
from langgraph.types import Command, interrupt  # pause and resume graph execution
from pydantic import BaseModel, Field  # define typed data models with validation

## Initialize one shared model

All three examples use the same model. Keeping that setup in one place makes it easier to compare the workflow patterns without changing the model each time.

In [ ]:
load_dotenv(".env")
model = init_chat_model("groq:openai/gpt-oss-20b", temperature=0)

## Typed output with `response_format`

Sometimes the next step needs clean data, not a paragraph. `response_format` helps the model return a result in a fixed structure that your code can use directly.

This is useful when the output needs to be routed, stored, or passed into another workflow step.

Structured output is most valuable when another system needs to consume the result: a ticket router, a database write, a dashboard, or another graph node. The model still writes the content, but LangChain validates that the final object matches the schema before exposing it as `structured_response`.


## Define the structured schema

The schema below defines the shape expected from the model.

This is where `Pydantic` matters: instead of asking for loose text, we ask for a result with named fields and fixed types.

In [ ]:
# Pydantic validates the final model output and gives application code a typed object.
class IncidentTriage(BaseModel):
    service: str = Field(description="The primary service involved in the incident")
    category: str = Field(description="A short routing label")
    priority: Literal["low", "medium", "high"]
    summary: str = Field(description="A concise operational summary")

## Build the structured-output agent

This agent has no tools. The important part here is `response_format=IncidentTriage`, which tells LangChain to return a validated object that matches that schema.

In [ ]:
structured_agent = create_agent(
    model=model,
    tools=[],
    system_prompt="Return structured incident triage data.",
    response_format=IncidentTriage,
)

## Run the classification request

The request below sends one incident description to the agent.

Notice what changes compared to a normal text response: the result includes a `structured_response` object instead of only free-form text.

In [ ]:
structured_result = structured_agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": (
                    "Database writes are timing out for paying customers in us-east-1. "
                    "Classify the incident."
                ),
            }
        ]
    }
)

# You can see that the output is the IncidentTriage object, which is exactly what we specified in the response_format.
structured_result["structured_response"]

Instead of prose, the agent returned a validated `IncidentTriage` object with the four fields above: a `service`, a `category`, a `priority`, and a one-line `summary`. The `service` and `category` fields are free text, so the exact wording the model chooses can vary between runs. The `priority` field, by contrast, is constrained to `low`, `medium`, or `high`, so any other value would have been rejected before your code saw it. This is why structured output is valuable at a system boundary: a ticket router or a database can consume the object directly.

## Convert the structured result to a plain dictionary

The previous cell returns a `Pydantic` object. `model_dump()` turns it into a regular Python dictionary, which is often the most convenient shape for application code.

In [ ]:
structured_result["structured_response"].model_dump()

## Streaming from graph nodes

Streaming is about seeing progress while the workflow is still running. Instead of waiting for one final answer, you can receive updates as each step finishes.

This is especially helpful for longer tasks because users can see that work is happening and what stage the system is in.

This example emits two kinds of data. Custom events describe progress in human-readable terms, while update events show the state fields each node returned. In a production interface, custom events often power status text and update events power debugging or observability views.


## Define the shared state for the streaming example

This `TypedDict` is the shared state for the graph.

- `topic` is the input
- `brief` will be filled in by the first node
- `risk` will be filled in by the second node

In [ ]:
class StreamState(TypedDict):
    topic: str
    brief: str
    risk: str

## Define the streaming nodes

This graph stays intentionally small:

- `draft_brief` creates the first useful piece of state: a short deployment summary
- `assess_risk` reads the same state and adds a simple risk level

Both also send short progress messages with `get_stream_writer()` so we can watch the workflow unfold step by step.

In [ ]:
def draft_brief(state: StreamState):
    writer = get_stream_writer()
    writer(
        {"stage": "draft_brief", "status": f"Preparing summary for {state['topic']}"}
    )
    return {"brief": f"Deployment summary for {state['topic']}"}


def assess_risk(state: StreamState):
    writer = get_stream_writer()
    writer({"stage": "assess_risk", "status": "Scoring operational risk"})
    risk = "high" if "payments" in state["topic"] else "medium"
    return {"risk": risk}

## Build the streaming graph

The graph below wires the two nodes together in a straight line:

`START -> draft_brief -> assess_risk -> END`

The rendered graph helps confirm that flow before we run it.

In [ ]:
stream_builder = StateGraph(cast(Any, StreamState))
stream_builder.add_node("draft_brief", draft_brief)
stream_builder.add_node("assess_risk", assess_risk)
stream_builder.add_edge(START, "draft_brief")
stream_builder.add_edge("draft_brief", "assess_risk")
stream_builder.add_edge("assess_risk", END)

stream_app: Any = stream_builder.compile()
display(Image(stream_app.get_graph(xray=True).draw_mermaid_png()))

## Stream the graph run

Now we run the graph in streaming mode.

`stream_mode=["custom", "updates"]` gives us both the custom progress messages and the state updates from each node.

In [ ]:
stream_input = cast(StreamState, {"topic": "payments migration"})

stream_events = list(
    stream_app.stream(
        stream_input,
        stream_mode=["custom", "updates"],
    )
)
stream_events

`stream_mode=["custom", "updates"]` returns two kinds of item. The `custom` events carry the human-readable progress messages the nodes wrote (`Preparing summary...`, then `Scoring operational risk`), while the `updates` events show each node's change to the state. Here `assess_risk` set `risk` to `high` because the topic contained "payments". A plain `invoke(...)` would return only the final state, without this step-by-step view.

## Compare with a regular invoke

The following invocation runs the same graph without streaming.

Compared to `stream(...)`, `invoke(...)` only gives you the final result after the graph is done.

In [ ]:
stream_app.invoke({"topic": "payments migration"})

## Interrupt-driven review

Sometimes the system should not continue on its own. For actions that are sensitive or risky, the workflow can pause and wait for a person to review the next step.

Once that decision comes back, the workflow resumes from the same point. This is useful when AI can prepare work, but a human should still make the final call.

An interrupt is not an exception or a failure. It is an intentional pause point. The graph stores its current checkpoint, returns an interrupt payload, and waits for the caller to resume the same thread with a decision.


## Define the approval state

This `TypedDict` holds the values that move through the approval workflow:

- the original request
- the draft plan prepared by the graph
- the reviewer decision
- the final status

In [ ]:
class ApprovalState(TypedDict, total=False):
    request: str
    draft: str
    approved: bool
    reviewer: str
    final_status: str

## Define the approval nodes

The workflow has three steps:

- `prepare_change` drafts the plan
- `review_change` pauses with `interrupt(...)`
- `finalize_change` builds the final status after the human decision comes back

This makes the pause-and-resume flow explicit in the graph itself.

In [ ]:
def prepare_change(state: ApprovalState):
    return {"draft": f"Prepared change plan for: {state['request']}"}


def review_change(state: ApprovalState):
    # Execution pauses here until the caller resumes this thread with a decision.
    decision = interrupt(
        {
            "question": "Approve the proposed change?",
            "draft": state["draft"],
        }
    )
    return {
        "approved": bool(decision.get("approved", False)),
        "reviewer": decision.get("reviewer", "unassigned"),
    }


def finalize_change(state: ApprovalState):
    status = "approved" if state["approved"] else "rejected"
    return {"final_status": f"{state['draft']} ({status} by {state['reviewer']})"}

## Build the approval graph

The graph below wires those three steps together and enables checkpoints with `InMemorySaver()`.

The checkpoints matter here because an interrupted workflow needs to pause and resume on the same `thread_id`.

In [ ]:
approval_builder = StateGraph(cast(Any, ApprovalState))
approval_builder.add_node("prepare_change", prepare_change)
approval_builder.add_node("review_change", review_change)
approval_builder.add_node("finalize_change", finalize_change)
approval_builder.add_edge(START, "prepare_change")
approval_builder.add_edge("prepare_change", "review_change")
approval_builder.add_edge("review_change", "finalize_change")
approval_builder.add_edge("finalize_change", END)

approval_app: Any = approval_builder.compile(checkpointer=InMemorySaver())
display(Image(approval_app.get_graph(xray=True).draw_mermaid_png()))

## Run until the workflow pauses

This first call starts the workflow, but it does not reach the end.

Execution stops at `interrupt(...)`, and the returned value contains the pause information.

In [ ]:
approval_config = cast(
    Any,
    {"configurable": {"thread_id": "approval-accepted"}},
)
paused = approval_app.invoke(
    {"request": "Rotate production credentials during the next maintenance window."},
    config=approval_config,
)
paused

## Inspect the interrupt payload

The following view extracts the interrupt details directly. A UI or external system could show this payload to a reviewer before resuming the workflow.

In [ ]:
paused["__interrupt__"]

The graph did not finish. It ran `prepare_change`, then stopped at `interrupt(...)` inside `review_change` and returned an `__interrupt__` payload with the question and the draft plan. Execution is paused, not failed: the checkpoint is saved and the graph waits for the caller to resume the same thread with a decision. A user interface could show this payload to a reviewer.

## Resume with approval

Now we resume the same workflow with a positive decision.

`Command(resume=...)` sends the human input back into the paused graph so it can continue from the interrupt point.

In [ ]:
resumed = approval_app.invoke(
    Command(resume={"approved": True, "reviewer": "ops-oncall"}),
    config=approval_config,
)
resumed

## Resume a second run with rejection

The same pattern can run on a different `thread_id`; this example resumes with a rejection decision.

Using a separate thread keeps the two approval runs independent.

In [ ]:
rejected_config = cast(
    Any,
    {"configurable": {"thread_id": "approval-rejected"}},
)
_ = approval_app.invoke(
    {"request": "Promote a schema change without a rollback plan."},
    config=rejected_config,
)

rejected = approval_app.invoke(
    Command(resume={"approved": False, "reviewer": "change-advisory"}),
    config=rejected_config,
)
rejected

Resuming with `Command(resume=...)` sends the human decision back into the paused graph, which continues from the interrupt point. The approved run finishes with a `final_status` ending `(approved by ops-oncall)`. The rejected run, on a separate `thread_id`, ends `(rejected by change-advisory)`. Using different threads keeps the two approvals independent.

## Inspect checkpoint history

Checkpoint history is the saved trail of what happened during a run. Looking back at it is useful for debugging, auditing, or understanding how the workflow reached its result.

Here we inspect the most recent few snapshots from the approved path.

In [ ]:
approval_history = list(approval_app.get_state_history(approval_config))[:3]
[
    {
        "step": (snapshot.metadata or {}).get("step"),
        "source": (snapshot.metadata or {}).get("source"),
        "next": snapshot.next,
        "values": snapshot.values,
    }
    for snapshot in approval_history
]

## Summary

In this notebook you:

- Implemented structured output with a Pydantic schema and read a validated `IncidentTriage` object.
- Applied streaming to watch custom progress events alongside per-node state updates.
- Built an interrupt that paused a graph for a human decision and resumed it with `Command`.
- Compared where each pattern fits: typed data at boundaries, streaming for visibility, interrupts for human control.

The optional follow-up notebooks show two ways to connect tools outside the local process: MCP servers and direct external API wrappers.

## References & Further Reading

- [**LangChain Structured Output**](https://docs.langchain.com/oss/python/langchain/structured-output): Returning typed, validated data from an agent.
- [**Streaming**](https://docs.langchain.com/oss/python/langgraph/streaming): The streaming modes for graph runs.
- [**Interrupts**](https://docs.langchain.com/oss/python/langgraph/interrupts): Pausing a graph and resuming with a decision.
- [**Human-in-the-loop**](https://docs.langchain.com/oss/python/langchain/human-in-the-loop): Patterns for human review of agent actions.